# Self-Supervised Learning: SimCLR, MoCo, MAE

**Module 4 - Notebook 6** | Skillsoft Track 2: DL Programmer

## Objectives
- SimCLR implementation on CIFAR-10
- Linear probe evaluation
- MoCo overview
- MAE (Masked Autoencoder) concept

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. SimCLR: Contrastive Learning Framework

In [ ]:
# SimCLR Augmentations
class SimCLRAugment:
    def __init__(self, size=32):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([
                transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.GaussianBlur(kernel_size=3),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
    
    def __call__(self, x):
        return self.transform(x), self.transform(x)

# NT-Xent Loss
def nt_xent_loss(z_i, z_j, temperature=0.5):
    """Normalized Temperature-scaled Cross Entropy Loss."""
    batch_size = z_i.size(0)
    
    # Normalize
    z_i = F.normalize(z_i, dim=1)
    z_j = F.normalize(z_j, dim=1)
    
    # Similarity matrix
    representations = torch.cat([z_i, z_j], dim=0)  # 2N x D
    similarity = F.cosine_similarity(representations.unsqueeze(1), representations.unsqueeze(0), dim=2)
    
    # Mask out self-similarity
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z_i.device)
    similarity = similarity.masked_fill(mask, -1e9)
    
    # Positive pairs: (i, i+N) and (i+N, i)
    positives = torch.cat([
        torch.diag(similarity, batch_size),
        torch.diag(similarity, -batch_size)
    ], dim=0)
    
    # Loss
    logits = similarity / temperature
    labels = torch.arange(2 * batch_size, device=z_i.device)
    labels[:batch_size] += batch_size
    labels[batch_size:] -= batch_size
    
    loss = F.cross_entropy(logits, labels)
    return loss

# SimCLR Model
class SimCLR(nn.Module):
    def __init__(self, base_encoder, projection_dim=128):
        super().__init__()
        self.encoder = base_encoder
        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )
    
    def forward(self, x):
        h = self.encoder(x)
        z = self.projection(h)
        return h, z

# ResNet-18 encoder (CIFAR version)
from torchvision.models import resnet18
base_encoder = resnet18(num_classes=512)
base_encoder.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
base_encoder.maxpool = nn.Identity()

model = SimCLR(base_encoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Data
train_dataset = torchvision.datasets.CIFAR10(
    './data', train=True, download=True,
    transform=SimCLRAugment()
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

# Train a few epochs
model.train()
for epoch in range(3):
    total_loss = 0
    for (x_i, x_j), _ in train_loader:
        x_i, x_j = x_i.to(device), x_j.to(device)
        optimizer.zero_grad()
        _, z_i = model(x_i)
        _, z_j = model(x_j)
        loss = nt_xent_loss(z_i, z_j)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}: NT-Xent Loss = {total_loss/len(train_loader):.4f}")

## 2. Linear Probe Evaluation

In [ ]:
# Freeze encoder, train linear classifier
for param in model.encoder.parameters():
    param.requires_grad = False

class LinearProbe(nn.Module):
    def __init__(self, encoder, num_classes=10):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x)
        return self.fc(h)

probe = LinearProbe(model.encoder).to(device)
optimizer_probe = torch.optim.Adam(probe.fc.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Standard CIFAR-10 train transform
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

probe_train_dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
probe_train_loader = DataLoader(probe_train_dataset, batch_size=256, shuffle=True, num_workers=2)

probe_test_dataset = torchvision.datasets.CIFAR10('./data', train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]))
probe_test_loader = DataLoader(probe_test_dataset, batch_size=256, shuffle=False, num_workers=2)

def train_probe(epoch):
    probe.train()
    for data, target in probe_train_loader:
        data, target = data.to(device), target.to(device)
        optimizer_probe.zero_grad()
        output = probe(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer_probe.step()

def eval_probe():
    probe.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in probe_test_loader:
            data, target = data.to(device), target.to(device)
            output = probe(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return correct / total

for epoch in range(10):
    train_probe(epoch)
    acc = eval_probe()
    print(f"Epoch {epoch}: Linear Probe Acc = {acc:.4f}")

## 3. MoCo v2/v3: Momentum Contrast (Executable Implementation)

MoCo (Momentum Contrast) — He et al., 2020; Chen et al., 2021 (v2/v3)

Key components implemented:
- **Momentum encoder**: EMA update of key encoder (m=0.999)
- **Queue**: Large dictionary of negative samples (65536)
- **MLP projection head**: 2-layer (v2/v3 improvement)
- **Strong augmentations**: SimCLR-style (v2)
- **Cosine annealing LR**: Learning rate schedule

In [ ]:
# MoCo v2/v3 Implementation
class MoCo(nn.Module):
    def __init__(self, base_encoder, dim=128, K=65536, m=0.999, T=0.07, mlp=True):
        super().__init__()
        self.K = K
        self.m = m
        self.T = T

        # Create encoders
        self.encoder_q = base_encoder(num_classes=dim)
        self.encoder_k = base_encoder(num_classes=dim)

        # MLP projection head (MoCo v2/v3)
        if mlp:
            dim_mlp = self.encoder_q.fc.weight.shape[1]
            self.encoder_q.fc = nn.Sequential(
                nn.Linear(dim_mlp, dim_mlp), nn.ReLU(), nn.Linear(dim_mlp, dim)
            )
            self.encoder_k.fc = nn.Sequential(
                nn.Linear(dim_mlp, dim_mlp), nn.ReLU(), nn.Linear(dim_mlp, dim)
            )

        # Initialize key encoder with query encoder weights
        for param_q, param_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            param_k.data.copy_(param_q.data)
            param_k.requires_grad = False

        # Queue
        self.register_buffer("queue", torch.randn(dim, K))
        self.queue = nn.functional.normalize(self.queue, dim=0)
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update_key_encoder(self):
        for param_q, param_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            param_k.data = param_k.data * self.m + param_q.data * (1. - self.m)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        batch_size = keys.shape[0]
        ptr = int(self.queue_ptr)
        assert self.K % batch_size == 0
        self.queue[:, ptr:ptr + batch_size] = keys.T
        ptr = (ptr + batch_size) % self.K
        self.queue_ptr[0] = ptr

    def forward(self, im_q, im_k):
        # Query encoder
        q = self.encoder_q(im_q)
        q = nn.functional.normalize(q, dim=1)

        # Key encoder
        with torch.no_grad():
            self._momentum_update_key_encoder()
            k = self.encoder_k(im_k)
            k = nn.functional.normalize(k, dim=1)

        # Positive logits: Nx1
        l_pos = torch.einsum('nc,nc->n', [q, k]).unsqueeze(-1)
        # Negative logits: NxK
        l_neg = torch.einsum('nc,ck->nk', [q, self.queue.clone().detach()])

        # Logits: Nx(1+K)
        logits = torch.cat([l_pos, l_neg], dim=1) / self.T
        labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)

        # Dequeue and enqueue
        self._dequeue_and_enqueue(k)

        loss = nn.CrossEntropyLoss()(logits, labels)
        return loss


# ResNet-50 encoder for MoCo (CIFAR adapted)
def resnet50_cifar(num_classes=128):
    from torchvision.models import resnet50
    model = resnet50(num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


# MoCo data augmentation (MoCo v2: stronger, SimCLR-style)
class MoCoAugment:
    def __init__(self, size=32):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.GaussianBlur(kernel_size=3),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
    def __call__(self, x):
        return self.transform(x), self.transform(x)


# Create MoCo model
moco_model = MoCo(resnet50_cifar, dim=128, K=65536, m=0.999, T=0.07, mlp=True).to(device)
moco_optimizer = torch.optim.SGD(moco_model.parameters(), lr=0.03, momentum=0.9, weight_decay=1e-4)

# Cosine annealing LR scheduler
moco_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(moco_optimizer, T_max=10)

# Data
moco_train_dataset = torchvision.datasets.CIFAR10(
    './data', train=True, download=True, transform=MoCoAugment()
)
moco_train_loader = DataLoader(moco_train_dataset, batch_size=256, shuffle=True, num_workers=2, drop_last=True)

print(f"MoCo parameters: {sum(p.numel() for p in moco_model.parameters()):,}")
print(f"Queue size: {moco_model.K}")

# Training loop (quick demo: 3 epochs)
moco_model.train()
for epoch in range(3):
    total_loss = 0
    for (im_q, im_k), _ in moco_train_loader:
        im_q, im_k = im_q.to(device), im_k.to(device)
        moco_optimizer.zero_grad()
        loss = moco_model(im_q, im_k)
        loss.backward()
        moco_optimizer.step()
        total_loss += loss.item()
    moco_scheduler.step()
    print(f"MoCo Epoch {epoch}: Loss = {total_loss/len(moco_train_loader):.4f}, LR = {moco_optimizer.param_groups[0]['lr']:.6f}")

## 4. MAE (Masked Autoencoder) — ViT Implementation (Executable)

MAE (Masked Autoencoder) — He et al., 2021

Key components implemented:
- **Patch embedding**: 16x16 patches with linear projection + positional encoding
- **High masking ratio**: 75% patches masked (only 25% visible to encoder)
- **Asymmetric architecture**: ViT encoder (visible patches only) + lightweight decoder
- **Reconstruction target**: Predict pixel values for masked patches (MSE loss)
- **Linear probe evaluation**: Frozen encoder + trained classifier

In [ ]:
# MAE: Patch Embedding
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, embed_dim=192):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)  # B, N, D
        return x


# MAE: Transformer Block
class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, bias=qkv_bias, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
        )
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0])
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


# MAE: Random Masking
def random_masking(x, mask_ratio=0.75):
    N, L, D = x.shape
    len_keep = int(L * (1 - mask_ratio))
    noise = torch.rand(N, L, device=x.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)
    ids_keep = ids_shuffle[:, :len_keep]
    x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
    mask = torch.ones([N, L], device=x.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)
    return x_masked, mask, ids_restore


# MAE Model
class MAE(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, embed_dim=192, depth=12, num_heads=3,
                 decoder_embed_dim=96, decoder_depth=4, decoder_num_heads=3, mlp_ratio=4.0):
        super().__init__()
        # Encoder
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        self.num_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Decoder
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim))
        self.decoder_blocks = nn.ModuleList([
            Block(decoder_embed_dim, decoder_num_heads, mlp_ratio) for _ in range(decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size ** 2 * in_chans)

        self.initialize_weights()

    def initialize_weights(self):
        torch.nn.init.trunc_normal_(self.pos_embed, std=0.02)
        torch.nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        torch.nn.init.trunc_normal_(self.cls_token, std=0.02)
        torch.nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_encoder(self, x, mask_ratio):
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]
        x, mask, ids_restore = random_masking(x, mask_ratio)
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)
        mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1)
        x_ = torch.cat([x[:, 1:, :], mask_tokens], dim=1)
        x_ = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x_.shape[2]))
        x = torch.cat([x[:, :1, :], x_], dim=1)
        x = x + self.decoder_pos_embed
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        x = self.decoder_pred(x)
        x = x[:, 1:, :]  # Remove cls token
        return x

    def forward_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        return loss

    def patchify(self, imgs):
        p = self.patch_size
        h = w = imgs.shape[2] // p
        x = imgs.reshape(imgs.shape[0], 3, h, p, w, p)
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(imgs.shape[0], h * w, p * p * 3)
        return x

    def forward(self, x, mask_ratio=0.75):
        latent, mask, ids_restore = self.forward_encoder(x, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(x, pred, mask)
        return loss, pred, mask


# Create MAE model (CIFAR-10: 32x32, patch_size=4 -> 8x8=64 patches)
mae_model = MAE(img_size=32, patch_size=4, in_chans=3, embed_dim=192, depth=12, num_heads=3,
                decoder_embed_dim=96, decoder_depth=4, decoder_num_heads=3).to(device)
mae_optimizer = torch.optim.AdamW(mae_model.parameters(), lr=1.5e-4, weight_decay=0.05)

# Data (CIFAR-10 for MAE)
mae_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
mae_train_dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=mae_transform)
mae_train_loader = DataLoader(mae_train_dataset, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

print(f"MAE parameters: {sum(p.numel() for p in mae_model.parameters()):,}")
print(f"Patches: {mae_model.num_patches} (patch_size={mae_model.patch_size})")

# Training loop (quick demo: 3 epochs)
mae_model.train()
for epoch in range(3):
    total_loss = 0
    for imgs, _ in mae_train_loader:
        imgs = imgs.to(device)
        mae_optimizer.zero_grad()
        loss, _, _ = mae_model(imgs, mask_ratio=0.75)
        loss.backward()
        mae_optimizer.step()
        total_loss += loss.item()
    print(f"MAE Epoch {epoch}: Loss = {total_loss/len(mae_train_loader):.6f}")

## 5. MAE Linear Probe Evaluation (Frozen Encoder + Classifier)

In [ ]:
# MAE Linear Probe: Freeze encoder, train classifier on CLS token
class MAELinearProbe(nn.Module):
    def __init__(self, mae_encoder, num_classes=10):
        super().__init__()
        self.encoder = mae_encoder
        self.fc = nn.Linear(mae_encoder.embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            x = self.encoder.patch_embed(x)
            x = x + self.encoder.pos_embed[:, 1:, :]
            cls_token = self.encoder.cls_token + self.encoder.pos_embed[:, :1, :]
            cls_tokens = cls_token.expand(x.shape[0], -1, -1)
            x = torch.cat([cls_tokens, x], dim=1)
            for blk in self.encoder.blocks:
                x = blk(x)
            x = self.encoder.norm(x)
            cls_out = x[:, 0]  # CLS token
        return self.fc(cls_out)


# Freeze MAE encoder
for param in mae_model.parameters():
    param.requires_grad = False

mae_probe = MAELinearProbe(mae_model).to(device)
mae_probe_optimizer = torch.optim.Adam(mae_probe.fc.parameters(), lr=1e-3, weight_decay=1e-4)
mae_probe_criterion = nn.CrossEntropyLoss()

# Data for linear probe
mae_probe_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
mae_probe_train_dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=mae_probe_transform)
mae_probe_train_loader = DataLoader(mae_probe_train_dataset, batch_size=256, shuffle=True, num_workers=2)

mae_probe_test_dataset = torchvision.datasets.CIFAR10('./data', train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]))
mae_probe_test_loader = DataLoader(mae_probe_test_dataset, batch_size=256, shuffle=False, num_workers=2)

def train_mae_probe(epoch):
    mae_probe.train()
    for data, target in mae_probe_train_loader:
        data, target = data.to(device), target.to(device)
        mae_probe_optimizer.zero_grad()
        output = mae_probe(data)
        loss = mae_probe_criterion(output, target)
        loss.backward()
        mae_probe_optimizer.step()

def eval_mae_probe():
    mae_probe.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in mae_probe_test_loader:
            data, target = data.to(device), target.to(device)
            output = mae_probe(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return correct / total

for epoch in range(10):
    train_mae_probe(epoch)
    acc = eval_mae_probe()
    print(f"MAE Linear Probe Epoch {epoch}: Acc = {acc:.4f}")

## 6. Summary & Key Takeaways

| Concept | Status | Notebook Feature |
|---------|--------|-------------------|
| SimCLR contrastive learning | ✅ Executable | NT-Xent loss + ResNet-18 + linear probe |
| **MoCo v2/v3** | ✅ **Executable (added)** | **Momentum encoder + queue (65K) + MLP head + cosine LR** |
| **MAE (Masked Autoencoder)** | ✅ **Executable (added)** | **ViT encoder + 75% masking + asymmetric decoder + linear probe** |
| Linear probe evaluation | ✅ Executable | Frozen encoder + trained classifier |

**Key Takeaways:**
1. **SimCLR** uses end-to-end contrastive learning with large batch sizes — effective but memory intensive.
2. **MoCo** decouples dictionary size from batch size using a momentum encoder and queue — enables larger negative sets with smaller batches.
3. **MAE** achieves SOTA with 75% masking ratio and asymmetric encoder/decoder — much more efficient than BERT-style 15% masking.
4. **All three benefit from stronger augmentations** (MoCo v2) and MLP projection heads.
5. **Linear probe accuracy** is the standard benchmark for representation quality — frozen encoder + trained classifier.